# Chapter 5
Use the `pennylane` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import pennylane as qml
from pennylane import numpy as np

n_qubits = 2
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.set_shots(None)
@qml.qnode(dev, interface="autograd", diff_method="backprop")
def quantum_score(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


rng = np.random.default_rng(7)
weight_shape = qml.StronglyEntanglingLayers.shape(
    n_layers=n_layers,
    n_wires=n_qubits,
)
initial_weights = np.array(
    0.05 * rng.normal(size=weight_shape),
    requires_grad=True,
)
x = np.array([0.25, -0.40], requires_grad=False)


def loss(weights):
    target = 1.0
    prediction = quantum_score(x, weights)
    return (prediction - target) ** 2


print("score:", quantum_score(x, initial_weights))
print("gradient shape:", qml.grad(loss)(initial_weights).shape)


In [ ]:
import pennylane as qml
from pennylane import numpy as np


def rotation_score(theta):
    """Measure Z after rotating |0> by theta radians about Y."""
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))


theta = np.array(0.7, requires_grad=True)
settings = [
    ("backprop", None),
    ("adjoint", None),
    ("parameter-shift", None),
    ("parameter-shift", 100),
    ("parameter-shift", 10000),
]
print("method          shots  mean(f)  sd(f)    mean(df) sd(df)")
for method, shots in settings:
    comparison_device = qml.device("default.qubit", wires=1, seed=7)
    node = qml.QNode(
        rotation_score, comparison_device,
        interface="autograd", diff_method=method,
    )
    node = qml.set_shots(node, shots=shots)
    derivative = qml.grad(node)
    values = np.array([float(node(theta)) for _ in range(64)])
    gradients = np.array([
        float(derivative(theta)) for _ in range(64)
    ])
    print(f"{method:15} {str(shots):5} "
          f"{values.mean(): .4f} {values.std(ddof=1):.4f}   "
          f"{gradients.mean(): .4f} {gradients.std(ddof=1):.4f}")


In [ ]:
import pennylane as qml
from pennylane import numpy as np


def make_interaction_data(n_samples=96, seed=11):
    rng = np.random.default_rng(seed)
    X = rng.uniform(low=-1.0, high=1.0, size=(n_samples, 2))
    y = np.where(X[:, 0] * X[:, 1] >= 0.0, 1.0, -1.0)
    X = (np.pi / 2) * X
    return np.array(X, requires_grad=False), np.array(y, requires_grad=False)


X, y = make_interaction_data()
X_train, y_train = X[:72], y[:72]
X_valid, y_valid = X[72:], y[72:]

n_qubits = 2
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.set_shots(None)
@qml.qnode(dev, interface="autograd", diff_method="backprop")
def circuit(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


def predict_one(x, weights, bias):
    return circuit(x, weights) + bias


def mse_loss(weights, bias, X_batch, y_batch):
    predictions = [predict_one(x, weights, bias) for x in X_batch]
    predictions = np.stack(predictions)
    return np.mean((predictions - y_batch) ** 2)


def accuracy(weights, bias, X_eval, y_eval):
    predictions = np.array([predict_one(x, weights, bias) for x in X_eval])
    predicted_labels = np.where(predictions >= 0.0, 1.0, -1.0)
    return np.mean(predicted_labels == y_eval)


rng = np.random.default_rng(3)
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers, n_qubits)
weights = np.array(0.1 * rng.normal(size=weight_shape), requires_grad=True)
bias = np.array(0.0, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=0.05)

for epoch in range(60):
    weights, bias = optimizer.step(
        lambda current_weights, current_bias: mse_loss(
            current_weights,
            current_bias,
            X_train,
            y_train,
        ),
        weights,
        bias,
    )

    if (epoch + 1) % 10 == 0:
        weight_gradient = qml.grad(mse_loss, argnums=0)(
            weights, bias, X_train, y_train
        )
        gradient_norm = np.linalg.norm(weight_gradient.ravel())
        train_loss = mse_loss(weights, bias, X_train, y_train)
        valid_acc = accuracy(weights, bias, X_valid, y_valid)
        print(f"epoch={epoch + 1:02d} loss={train_loss:.3f} "
              f"valid_acc={valid_acc:.3f} grad_norm={gradient_norm:.3f}")


In [ ]:
weight_gradient = qml.grad(mse_loss, argnums=0)(
    weights, bias, X_train, y_train
)
gradient_norm = np.linalg.norm(weight_gradient.ravel())
print("gradient norm:", float(gradient_norm))


In [ ]:
@qml.set_shots(None)
@qml.qnode(dev, interface="autograd", diff_method="backprop")
def quantum_features(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


def hybrid_prediction(x, weights, head_weights, head_bias):
    features = np.stack(quantum_features(x, weights))
    return np.dot(head_weights, features) + head_bias


In [ ]:
import pennylane as qml
from pennylane import numpy as np


def make_regression_data(n_samples=80, seed=13):
    rng = np.random.default_rng(seed)
    X = rng.uniform(low=-1.0, high=1.0, size=(n_samples, 2))
    angles = np.pi * X
    y = 0.6 * np.sin(angles[:, 0]) + 0.3 * np.cos(2.0 * angles[:, 1])
    return np.array(angles, requires_grad=False), np.array(y, requires_grad=False)


X, y = make_regression_data()
X_train, y_train = X[:60], y[:60]
X_valid, y_valid = X[60:], y[60:]

n_qubits = 2
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.set_shots(None)
@qml.qnode(dev, interface="autograd", diff_method="backprop")
def regressor_circuit(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


def predict(x, weights, scale, bias):
    return scale * regressor_circuit(x, weights) + bias


def regression_loss(weights, scale, bias, X_batch, y_batch):
    predictions = np.stack([predict(x, weights, scale, bias) for x in X_batch])
    return np.mean((predictions - y_batch) ** 2)


rng = np.random.default_rng(5)
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers, n_qubits)
weights = np.array(0.05 * rng.normal(size=weight_shape), requires_grad=True)
scale = np.array(1.0, requires_grad=True)
bias = np.array(0.0, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=0.03)

for epoch in range(100):
    weights, scale, bias = optimizer.step(
        lambda current_weights, current_scale, current_bias: regression_loss(
            current_weights,
            current_scale,
            current_bias,
            X_train,
            y_train,
        ),
        weights,
        scale,
        bias,
    )

valid_mse = regression_loss(weights, scale, bias, X_valid, y_valid)
print("validation MSE:", float(valid_mse))


In [ ]:
def trig_features(X):
    columns = [np.ones(len(X))]
    for j in range(X.shape[1]):
        columns.extend(
            [
                np.sin(X[:, j]),
                np.cos(X[:, j]),
                np.sin(2.0 * X[:, j]),
                np.cos(2.0 * X[:, j]),
            ]
        )
    return np.stack(columns, axis=1)


def fit_ridge(Phi, y, ridge=1e-3):
    gram = Phi.T @ Phi
    penalty = ridge * np.eye(gram.shape[0])
    penalty[0, 0] = 0.0
    return np.linalg.solve(gram + penalty, Phi.T @ y)


Phi_train = trig_features(X_train)
Phi_valid = trig_features(X_valid)
coefficients = fit_ridge(Phi_train, y_train)
baseline_predictions = Phi_valid @ coefficients
baseline_mse = np.mean((baseline_predictions - y_valid) ** 2)
print("baseline validation MSE:", float(baseline_mse))


In [ ]:
import pennylane as qml
import torch

torch.manual_seed(7)
n_qubits = 2
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.set_shots(None)
@qml.qnode(dev, interface="torch", diff_method="backprop")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


weight_shapes = {"weights": (n_layers, n_qubits, 3)}
quantum_layer = qml.qnn.TorchLayer(qnode, weight_shapes)

model = torch.nn.Sequential(
    quantum_layer,
    torch.nn.Linear(n_qubits, 1),
    torch.nn.Tanh(),
)
